# 02 — Emissions

Builds the single emissions table that notebook 03 divides by each country's
carbon budget.

Two sources, because neither covers the ground alone:

| Countries | Source | Why |
|---|---|---|
| Pacific islands | SPC **Pacific Data Hub** | The Pacific's own statistics office, reconciled with national inventories |
| Everyone else | **Our World in Data** (EDGAR / GCP) | Consistent global coverage |

The Pacific Data Hub takes priority wherever both have a country. That matters
more than it sounds: OWID's `total_ghg` puts French Polynesia at 68 Mt CO2-eq
for 280,000 people — roughly 240 tonnes each, which is not a real number. Its
land-use-change component swamps these small territories. The Pacific Data Hub
figure is 0.8 Mt.

**Outputs**
- `asr/A_lca/.../merged__gwp100_lcia.csv` — the ASR numerator, for pyaesa
- `data_viz/emissions.csv`, `data_viz/countries.csv` — for the D3 app

In [ ]:
import io

import pandas as pd

from config import (
    LCA_FILE, PACIFIC, PACIFIC_ISO3, REGION_COL, VIZ, WB_POP, YEARS, YEAR_COLS,
)
from pdh_api import fetch_data_pacific

## 1. World — Our World in Data

`total_ghg` is all greenhouse gases in million tonnes CO2-eq, wrapping EDGAR
and the Global Carbon Project.

In [2]:
OWID_URL = "https://github.com/owid/co2-data/raw/master/owid-co2-data.csv"

try:
    owid = pd.read_csv(OWID_URL)
except Exception as exc:
    # python.org builds on macOS ship without CA certificates until you run
    # /Applications/Python\ 3.x/Install\ Certificates.command
    print(f"Direct read failed ({type(exc).__name__}); retrying without TLS verification.")
    import requests
    import urllib3

    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    owid = pd.read_csv(io.StringIO(requests.get(OWID_URL, verify=False).text))

print(f"{len(owid):,} rows x {owid.shape[1]} columns")

Direct read failed (URLError); retrying without TLS verification.
50,411 rows x 79 columns


In [3]:
# Real countries only: OWID mixes aggregates like "OWID_EUR" into iso_code.
world = owid[
    owid["year"].isin(YEARS)
    & owid["iso_code"].notna()
    & ~owid["iso_code"].str.startswith("OWID")
].copy()

names = world.drop_duplicates("iso_code").set_index("iso_code")["country"]

# A few countries have interior gaps in total_ghg. Carry the nearest
# observation across them rather than dropping the country outright.
world = world.sort_values(["iso_code", "year"])
world["total_ghg"] = world.groupby("iso_code")["total_ghg"].ffill().bfill()
world = world.dropna(subset=["total_ghg"])

# OWID reports million tonnes CO2-eq; pyaesa expects kg.
world["emissions_kg"] = world["total_ghg"] * 1e9
world = world[["iso_code", "year", "emissions_kg"]]

print(f"{world['iso_code'].nunique()} countries")

218 countries


## 2. Pacific — SPC Pacific Data Hub

`DF_CLIMATE_CHANGE` reports GHG emissions per capita in tonnes CO2-eq, so it
needs multiplying by population to get national totals. Population comes from
the World Bank table pyaesa processed in notebook 01 — the same numbers that
set each country's carbon budget, so numerator and denominator agree.

In [4]:
ghg = fetch_data_pacific(
    source="DF_CLIMATE_CHANGE",
    start_period=str(min(YEARS)),
    end_period=str(max(YEARS)),
    key="A.GHG_EMI_CAPITA." + "+".join(PACIFIC),
)

print(ghg.groupby("GEO_PICT")["TIME_PERIOD"].agg(["min", "max", "count"]))

           min   max  count
GEO_PICT                   
FJ        2000  2023     24
FM        2000  2023     24
KI        2000  2023     24
MH        2000  2023     24
NC        2000  2023     24
NR        2000  2023     24
PF        2000  2023     24
PG        2000  2023     24
PW        2000  2023     24
SB        2000  2023     24
TO        2000  2023     24
TV        2000  2023     24
VU        2000  2023     24
WS        2000  2023     24


In [5]:
population = (
    pd.read_csv(WB_POP)
    .query("variable == 'Population'")
    .melt(id_vars="iso3_code", value_vars=YEAR_COLS,
          var_name="year", value_name="population")
    .astype({"year": int})
    .rename(columns={"iso3_code": "iso_code"})
)

pacific = ghg.assign(
    iso_code=ghg["GEO_PICT"].map(PACIFIC),
    year=ghg["TIME_PERIOD"].astype(int),
).dropna(subset=["iso_code"])

pacific = pacific.merge(population, on=["iso_code", "year"])

# SPC reports tonnes CO2-eq per person; pyaesa expects kg.
pacific["emissions_kg"] = pacific["value"] * pacific["population"] * 1_000
pacific = pacific[["iso_code", "year", "emissions_kg"]]

print(f"{pacific['iso_code'].nunique()} islands, {len(pacific)} country-years")

14 islands, 336 country-years


## 3. Merge

Pacific Data Hub wins wherever it has a country.

In [6]:
emissions = pd.concat([
    world[~world["iso_code"].isin(PACIFIC_ISO3)].assign(source="OWID"),
    pacific.assign(source="PDH"),
], ignore_index=True)

# Inner join on population both attaches per-capita figures and drops
# countries the World Bank has no entry for — pyaesa cannot allocate a carbon
# budget to those, so they cannot get an ASR either.
emissions = emissions.merge(population, on=["iso_code", "year"])
emissions["emissions_t_per_capita"] = (
    emissions["emissions_kg"] / emissions["population"] / 1_000
)

print(emissions.groupby("source")["iso_code"].nunique())

source
OWID    192
PDH      14
Name: iso_code, dtype: int64


## 4. Write outputs

pyaesa reads a wide table keyed by `r_p` — the producing region, since this is
production-based accounting (PBA): emissions counted where they physically
happen, not where the goods they embody are consumed. That's what both the
Pacific Data Hub and OWID report; using `r_p` (fu_code `L1.b`) keeps the label
honest. Countries missing any year are dropped: the ASR step needs every
budget row to find exactly one emissions row.

In [ ]:
lca = (
    emissions
    .pivot(index="iso_code", columns="year", values="emissions_kg")
    .reindex(columns=list(YEARS))
    .dropna()
)
lca.columns = [str(c) for c in lca.columns]
lca = lca.reset_index().rename(columns={"iso_code": REGION_COL})
lca.insert(1, "impact", "GWP_100")
lca.insert(2, "impact_unit", "kg CO2-eq")

LCA_FILE.parent.mkdir(parents=True, exist_ok=True)
lca.to_csv(LCA_FILE, index=False)

print(f"{len(lca)} countries -> {LCA_FILE.relative_to(LCA_FILE.parents[4])}")

In [ ]:
VIZ.mkdir(exist_ok=True)
kept = emissions[emissions["iso_code"].isin(lca[REGION_COL])]

kept.to_csv(VIZ / "emissions.csv", index=False)

countries = (
    kept.sort_values("year")
    .groupby("iso_code")
    .agg(population=("population", "last"))
    .reset_index()
    .assign(
        name=lambda d: d["iso_code"].map(names),
        is_pacific=lambda d: d["iso_code"].isin(PACIFIC_ISO3),
        source=lambda d: d["iso_code"].map(
            lambda c: "PDH" if c in PACIFIC_ISO3 else "OWID"
        ),
    )[["iso_code", "name", "population", "is_pacific", "source"]]
)
countries.to_csv(VIZ / "countries.csv", index=False)

print(f"{len(countries)} countries, {countries['is_pacific'].sum()} Pacific")